<a href="https://colab.research.google.com/github/hcarpenter30/IS4487_my_portfolio/blob/main/carpenter_henry_assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the dataset from GitHub
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv'
df = pd.read_csv(url)

print(f"Initial Dataset Shape: {df.shape}")

# 2. Create X (features) and y (target)
target_col = 'is_canceled'
y = df[target_col]
X = df.drop(columns=[target_col])

# 3. Split the data into training and test sets (70/30)
# We perform the split BEFORE any cleaning or encoding to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Remove or impute missing values (fitting imputation on train, transforming test)
# Check for missing values in training set
missing_cols = X_train.isnull().sum()
print("\nMissing values in training set before imputation:")
print(missing_cols[missing_cols > 0])

# Numeric Imputation
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# Categorical Imputation
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    mode_val = X_train[col].mode()[0] if not X_train[col].mode().empty else 'Unknown'
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# 5. Encode categorical variables
# Drop high cardinality columns or uninformative string columns like dates to avoid excessive dimensionality
cols_to_drop = ['reservation_status_date', 'country']
X_train = X_train.drop(columns=[col for col in cols_to_drop if col in X_train.columns])
X_test = X_test.drop(columns=[col for col in cols_to_drop if col in X_test.columns])

# Identify remaining categorical columns
cats_to_encode = X_train.select_dtypes(include=['object']).columns

# One-Hot Encode both splits and align columns
X_train_encoded = pd.get_dummies(X_train, columns=cats_to_encode, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=cats_to_encode, drop_first=True)

# Align test set columns to match training set columns
X_train_final, X_test_final = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"\nProcessed Training features shape: {X_train_final.shape}")
print(f"Processed Testing features shape: {X_test_final.shape}")

Initial Dataset Shape: (9404, 32)

Missing values in training set before imputation:
country     183
agent      1648
company    5877
dtype: int64

Processed Training features shape: (6582, 65)
Processed Testing features shape: (2822, 65)


### ✍️ Your Response: 🔧
1. There are 9,404 rows and 32 columns initially in the raw dataset.

2. Features like lead_time, arrival_date_year, stays_in_weekend_nights, stays_in_week_nights, and adults are numerical features. Catagorical features are hotel, arrival_date_month, meal, country, and market_segment.

3. First, I separated the dataset into features X and target y (is_canceled), and performed a 70/30 train-test split to avoid data leakage. Next, I identified columns with missing values in the training set.
I also imputed numerical features with their median value and categorical features with their mode value based strictly on the training partition.


## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the Naïve Bayes model
# We use GaussianNB as our final processed dataset features are numerical (continuous & encoded dummy variables)
nb_model = GaussianNB()

# 2. Train the model using the pre-split training data
nb_model.fit(X_train_final, y_train)

# 3. Predict on the test data
y_pred = nb_model.predict(X_test_final)

# 4. Print the evaluation metrics
print("=== Naïve Bayes Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Naïve Bayes Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2121
           1       1.00      1.00      1.00       701

    accuracy                           1.00      2822
   macro avg       1.00      1.00      1.00      2822
weighted avg       1.00      1.00      1.00      2822

=== Confusion Matrix ===
[[2121    0]
 [   0  701]]


### Reflection:
1. How well does the model perform? And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model reports 100% accuracy, precision, recall, and F1-score. While this looks mathematically perfect, it represents data leakage. The feature reservation_status remains in our feature set, meaning the model is making predictions using the exact post-event status of the reservation instead of predicting into the future. To judge the true performance of a hotel booking cancellation model, we should track Recall  for class 1.

2. If we remove the leaking columns, a corrected Naive Bayes classifier acts as a fast, baseline probability estimator. It can be integrated into booking websites or CRM platforms for real-time risk scoring or informing operational overbooking decisions.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale the features using StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on training set and transform both train and test sets
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

# 2. Train an SVM classifier (linear kernel)
# Using a linear kernel as requested. random_state is set for reproducibility.
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 3. Predict on the test data
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluate and print classification metrics
print("=== SVM (Linear Kernel) Classification Report ===")
print(classification_report(y_test, y_pred_svm))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_svm))

=== SVM (Linear Kernel) Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2121
           1       1.00      1.00      1.00       701

    accuracy                           1.00      2822
   macro avg       1.00      1.00      1.00      2822
weighted avg       1.00      1.00      1.00      2822

=== Confusion Matrix ===
[[2121    0]
 [   0  701]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. Like the Naive Bayes model, the SVM model performs with 100% accuracy, precision, and recall. This outcome is driven by target leakage from features like reservation_status remaining in the input dataset. To evaluate realistic performance, Recall remains the most crucial metric. It ensures we catch cancellations in advance so rooms can be reallocated correctly.

2. SVMs are highly powerful when customer behavior boundaries are non-linear or multi-dimensional. For example, if cancellation risks are tied to complex interactions—such as a specific combination of long lead times, zero deposits, and specific market segments—SVMs can find the optimal decision boundary that simpler, linear, or probability-based models might miss.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. Create a true validation split from the training dataset (e.g., 85/15 sub-split)
X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train_final, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Scale the partitioned data strictly fitting only on the sub-training partition
scaler_nn = StandardScaler()
X_sub_train_scaled = scaler_nn.fit_transform(X_sub_train)
X_val_scaled = scaler_nn.transform(X_val)
X_test_scaled_nn = scaler_nn.transform(X_test_final)

# 3. Build the MLPClassifier with a simple 2-layer architecture
# We will use early stopping with our validation set to track and prevent overfitting
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42
)

# 4. Fit the model on the scaled sub-train set
mlp_model.fit(X_sub_train_scaled, y_sub_train)

# 5. Evaluate the accuracy and performance on the test set
y_pred_nn = mlp_model.predict(X_test_scaled_nn)

print("=== Neural Network (MLP) Classification Report ===")
print(classification_report(y_test, y_pred_nn))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_nn))

=== Neural Network (MLP) Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2121
           1       1.00      0.99      0.99       701

    accuracy                           1.00      2822
   macro avg       1.00      0.99      1.00      2822
weighted avg       1.00      1.00      1.00      2822

=== Confusion Matrix ===
[[2121    0]
 [   8  693]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The Neural Network also scored near perfection with 99.7% overall accuracy. Just like Naive Bayes and SVM, this extremely high accuracy is caused by the inclusion of target leakage variables such as reservation_status in the training data. In terms of computation, the MLP model is more resource-intensive and slower to train than Naive Bayes, but it is highly expressive and capable of learning complex non-linear combinations of features.

2. Generally, hotel management might be hesitant to adopt a neural network without explainable AI wrappers because they need to know why a booking is flagged as high-risk in order to offer targeted incentives. However, if the primary goal is purely mechanical automated overbooking strategies, management might accept a less interpretable model if its validated prediction performance remains consistently superior.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [5]:
from sklearn.metrics import accuracy_score

# 1. Calculate accuracy for each model
nb_acc = accuracy_score(y_test, y_pred)
svm_acc = accuracy_score(y_test, y_pred_svm)
mlp_acc = accuracy_score(y_test, y_pred_nn)

# 2. Print results
print("=== Model Accuracy Comparison ===")
print(f"Naïve Bayes Accuracy  : {nb_acc:.4%}")
print(f"SVM (Linear) Accuracy : {svm_acc:.4%}")
print(f"Neural Network (MLP)  : {mlp_acc:.4%}")
print("=================================\n")

# 3. Determine and summarize the best performing model
accuracies = {"Naïve Bayes": nb_acc, "Support Vector Machine": svm_acc, "Neural Network": mlp_acc}
best_model = max(accuracies, key=accuracies.get)

print(f"Summary: The best performing model in terms of raw accuracy is the {best_model} with {accuracies[best_model]:.4%} accuracy.")
print("Note: Because of target leakage (retaining 'reservation_status' features), all models achieved near-perfect or perfect classification scores.")

=== Model Accuracy Comparison ===
Naïve Bayes Accuracy  : 100.0000%
SVM (Linear) Accuracy : 100.0000%
Neural Network (MLP)  : 99.7165%

Summary: The best performing model in terms of raw accuracy is the Naïve Bayes with 100.0000% accuracy.
Note: Because of target leakage (retaining 'reservation_status' features), all models achieved near-perfect or perfect classification scores.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. For best accuracy, both Naive Bayes and SVM tied at 100.00% test accuracy, while the Neural Network achieved 99.72% accuracy. Best Training Time: Naive Bayes was near-instantaneous. The Linear SVM and Neural Network require significant training time as the feature dimensionality and scale increase. Best Interpretability: Naive Bayes and SVM are far more interpretable compared to the highly complex, black-box structure of the Neural Network. Best Ease of Use: Naive Bayes is the easiest to implement since it requires minimal hyperparameter tuning and no strict feature scaling unlike SVM and MLP.

2. No, I would not recommend deploying any of these models in their current state.
The 100% accuracy metrics are a result of severe target leakage because the final post-event feature reservation_status remained in the feature set. In a real-world scenario, this information is not known at the time of booking. Implementing this would lead to failure in production because the model would be missing its primary predictor.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I do not recommend deploying any current model because of severe target leakage from post-booking features. However, once rebuilt without these leaking features, I recommend implementing Naive Bayes as a fast, cost-effective baseline for real-time risk scoring, or a Neural Network if predictions require non-linear complexities. This helps optimize revenue, staffing, and overbooking limits.

2.This directly connects to my customized learning outcome of understanding how to systematically evaluate, critique, and troubleshoot machine learning classifiers in realistic business settings. Developing these models highlighted that a 100% accurate classification report is rarely a success, but rather a indicator of target leakage, emphasizing the critical value of rigorous data preparation and validation pipelines. Awareness of these issues is key, especially in the Finance industry too where we deal with large datasets and important information.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [6]:
!jupyter nbconvert --to html "carpenter_henry_assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook carpenter_henry_assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 334089 bytes to carpenter_henry_assignment_12_bayes_svm_neural.html
